# NLP Genre Classification Project [cite: 1]
## Complete Sequential Roadmap [cite: 2]
**Environment**: Python 3.12 | Local Machine (CPU) → Kaggle (GPU) [cite: 3]

**Instructions**: Every phase is meant to be run in order, top to bottom. Each phase specifies where to run it (local or Kaggle), the required input files, and the output files it produces. Never skip ahead. [cite: 4]

## PHASE 1 — Data Preparation [cite: 9]
**Run On**: LOCAL MACHINE (CPU) | No GPU needed [cite: 11]

Both datasets use different genre tag formats. All genres must be normalised into the same 12-label vocabulary, texts cleaned, and data split proportionally. [cite: 10]

### 1.1 — Download Datasets [cite: 12]
Download both datasets to your project folder before running any code. [cite: 14]
- **CMU Book Summary Dataset**: `booksummaries.txt` from http://www.cs.cmu.edu/~dbamman/booksummaries.html [cite: 15, 16, 17, 18]
- **TMDB Movies Dataset**: `movies_metadata.csv` from https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset [cite: 19, 20, 21, 22]

### 1.2 — Parse CMU Book Genres [cite: 23]
Extracts human-readable names into a plain Python list that the mapping step can work with. [cite: 24]

In [5]:
import pandas as pd
import ast

cmu = pd.read_csv("booksummaries.txt", sep="\t", header=None,
    names=["wiki_id","freebase_id","title","author","pub_date","genres","summary"])

# Diagnostic: inspect raw genre format before parsing
print(repr(cmu["genres"].dropna().iloc[0]))  # See exact format

def parse_cmu_genres(genre_str):
    # Uses ast.literal_eval, NOT json.loads, because:
    # 1. CMU genres are Python dict literals with single quotes
    # 2. Genre names can contain apostrophes (e.g. Children's literature)
    #    which break the single-quote replace hack used with json.loads
    # 3. Older CMU entries have Python 2 u'..' prefix, rejected by json
    if pd.isna(genre_str) or genre_str.strip() == "{}":
        return []
    try:
        return list(ast.literal_eval(genre_str).values())
    except:
        return []

cmu["genre_list"] = cmu["genres"].apply(parse_cmu_genres)
print(f"CMU rows: {len(cmu)}")  # ~16,559


'{"/m/016lj8": "Roman \\u00e0 clef", "/m/06nbt": "Satire", "/m/0dwly": "Children\'s literature", "/m/014dfn": "Speculative fiction", "/m/02xlf": "Fiction"}'
CMU rows: 16559


### 1.3 — Parse TMDB Movie Genres [cite: 38]
Extract plain genre name lists so both datasets are in the same format before mapping. [cite: 39]

In [6]:
import ast

movies = pd.read_csv("movies_metadata.csv", low_memory=False)
movies = movies[["title", "overview", "genres"]].dropna()

def parse_tmdb_genres(genre_str):
    if pd.isna(genre_str): return []
    try: return [g["name"] for g in ast.literal_eval(genre_str)]
    except: return []

movies["genre_list"] = movies["genres"].apply(parse_tmdb_genres)
movies = movies[movies["overview"].str.len() > 30]
print(f"Movies rows: {len(movies)}")  # ~40,000


Movies rows: 44253


### 1.4 — Genre Taxonomy Mapping [cite: 50]
Maps the raw genre strings to 12 shared target genres. The lowercase strip is essential due to inconsistent casing. [cite: 51, 52, 53]

In [7]:
# Complete genre map — keys are ALL lowercase
GENRE_MAP = {
    "action": "Action", "adventure": "Action", "action/adventure": "Action",
    "spy film": "Action", "superhero": "Action", "martial arts film": "Action",
    "comedy": "Comedy", "romantic comedy": "Comedy", "satire": "Comedy",
    "black comedy": "Comedy", "parody": "Comedy", "humour": "Comedy",
    "drama": "Drama", "melodrama": "Drama", "historical fiction": "Drama",
    "war film": "Drama", "biographical film": "Drama", "legal drama": "Drama",
    "historical novel": "Drama", "historical drama": "Drama",
    "horror": "Horror", "horror fiction": "Horror", "gothic fiction": "Horror",
    "psychological horror": "Horror", "supernatural fiction": "Horror",
    "thriller": "Thriller", "crime fiction": "Thriller", "mystery": "Thriller",
    "detective fiction": "Thriller", "crime": "Thriller", "suspense": "Thriller",
    "noir fiction": "Thriller", "psychological thriller": "Thriller",
    "science fiction": "Sci-Fi", "cyberpunk": "Sci-Fi", "space opera": "Sci-Fi",
    "dystopia": "Sci-Fi", "dystopian fiction": "Sci-Fi", "hard science fiction": "Sci-Fi",
    "alternate history": "Sci-Fi",
    "fantasy": "Fantasy", "high fantasy": "Fantasy", "epic fantasy": "Fantasy",
    "urban fantasy": "Fantasy", "dark fantasy": "Fantasy", "fairy tale": "Fantasy",
    "romance novel": "Romance", "romance": "Romance", "love story": "Romance",
    "chick lit": "Romance", "romantic fiction": "Romance",
    "animation": "Animation", "animated film": "Animation", "family film": "Animation",
    "documentary": "Documentary", "docudrama": "Documentary",
    "literary fiction": "Literary Fiction", "fiction": "Literary Fiction",
    "bildungsroman": "Literary Fiction", "philosophical fiction": "Literary Fiction",
    "non-fiction": "Non-Fiction", "biography": "Non-Fiction",
    "autobiography": "Non-Fiction", "memoir": "Non-Fiction",
}

TARGET_GENRES = ["Action","Comedy","Drama","Horror","Thriller",
                 "Sci-Fi","Fantasy","Romance","Animation",
                 "Documentary","Literary Fiction","Non-Fiction"]

def map_genres(raw_list):
    mapped = set()
    for g in raw_list:
        key = g.strip().lower()  # lowercase before lookup — fixes casing issues
        if key in GENRE_MAP:
            mapped.add(GENRE_MAP[key])
    return list(mapped)

cmu["mapped_genres"]    = cmu["genre_list"].apply(map_genres)
movies["mapped_genres"] = movies["genre_list"].apply(map_genres)

cmu    = cmu[cmu["mapped_genres"].apply(len) > 0].reset_index(drop=True)
movies = movies[movies["mapped_genres"].apply(len) > 0].reset_index(drop=True)

print(f"CMU after mapping:    {len(cmu)}")    # ~9,000-10,000
print(f"Movies after mapping: {len(movies)}")  # ~30,000+


CMU after mapping:    10163
Movies after mapping: 41473


### 1.5 — Save Cleaned DataFrames [cite: 98]
Saves original datasets paired by title for mismatch detection later. [cite: 99, 100]

In [8]:
# Save cleaned dataframes for use in Phase 7
cmu[["title","summary","mapped_genres"]].to_csv("cmu_cleaned.csv", index=False)
movies[["title","overview","mapped_genres"]].to_csv("movies_cleaned.csv", index=False)
print("Saved: cmu_cleaned.csv")
print("Saved: movies_cleaned.csv")

# Files produced so far:
# cmu_cleaned.csv     <- needed in Phase 7
# movies_cleaned.csv  <- needed in Phase 7


Saved: cmu_cleaned.csv
Saved: movies_cleaned.csv


### 1.6 — Multi-Label Binarization and Train/Val/Test Split [cite: 110]
Uses `MultiLabelBinarizer` and `MultilabelStratifiedShuffleSplit`. Do not re-run if already completed locally. [cite: 111, 112, 113]

In [9]:
# YOU HAVE ALREADY RUN THIS — DO NOT RE-RUN
# Included for reference only

from sklearn.preprocessing import MultiLabelBinarizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import numpy as np, pickle

mlb = MultiLabelBinarizer(classes=TARGET_GENRES)
mlb.fit([TARGET_GENRES])

cmu_df = pd.DataFrame({
    "text":       cmu["summary"].values,
    "source":     "book",
    "title":      cmu["title"].values,
    "labels":     list(mlb.transform(cmu["mapped_genres"]))
})
movie_df = pd.DataFrame({
    "text":   movies["overview"].values,
    "source": "movie",
    "title":  movies["title"].values,
    "labels": list(mlb.transform(movies["mapped_genres"]))
})
full_df = pd.concat([cmu_df, movie_df], ignore_index=True)
full_df["text"] = full_df["text"].str.strip().str.replace(r"\s+", " ", regex=True)
full_df = full_df[full_df["text"].str.len() > 30]

X = full_df["text"].values
y = np.stack(full_df["labels"].values)

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
for tv_idx, test_idx in msss.split(X, y):
    X_tv, X_test = X[tv_idx], X[test_idx]
    y_tv, y_test = y[tv_idx], y[test_idx]

msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.176, random_state=42)
for train_idx, val_idx in msss2.split(X_tv, y_tv):
    X_train, X_val = X_tv[train_idx], X_tv[val_idx]
    y_train, y_val = y_tv[train_idx], y_tv[val_idx]

with open("data_splits.pkl", "wb") as f:
    pickle.dump({"X_train":X_train,"y_train":y_train,
                 "X_val":X_val,"y_val":y_val,
                 "X_test":X_test,"y_test":y_test}, f)
with open("mlb.pkl", "wb") as f:
    pickle.dump(mlb, f)
print("Saved: data_splits.pkl")
print("Saved: mlb.pkl")


Saved: data_splits.pkl
Saved: mlb.pkl


## PHASE 2 — Generate Embeddings (Topic Modeling Prep) [cite: 155]
**Run On**: LOCAL MACHINE (CPU) [cite: 157]
Uses SBERT `all-MiniLM-L6-v2` to avoid re-encoding on Kaggle. Needed by BERTopic and Mismatch Detection. [cite: 156, 159]

In [10]:
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer

# Load splits
with open("data_splits.pkl", "rb") as f:
    splits = pickle.load(f)
X_train = splits["X_train"]
X_val   = splits["X_val"]
X_test  = splits["X_test"]

# Combine ALL texts in this fixed order — order must never change
# all_embeddings[0:len(X_train)] = train embeddings
# all_embeddings[len(X_train):len(X_train)+len(X_val)] = val embeddings
# all_embeddings[len(X_train)+len(X_val):] = test embeddings
all_texts = list(X_train) + list(X_val) + list(X_test)
print(f"Total texts to embed: {len(all_texts)}")

# Load SBERT model (~80 MB download on first run)
sbert = SentenceTransformer("all-MiniLM-L6-v2")

# Encode — batch_size=64 works on CPU
# On a mid-range laptop this takes roughly 30-50 minutes
embeddings = sbert.encode(
    all_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"Embeddings shape: {embeddings.shape}")  # Should be (N, 384)
assert len(all_texts) == len(embeddings), "Count mismatch!"
np.save("all_embeddings.npy", embeddings)
print("Saved: all_embeddings.npy")


c:\Users\ab\Desktop\NLP_Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total texts to embed: 51618


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5244.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 807/807 [01:50<00:00,  7.29it/s]


Embeddings shape: (51618, 384)
Saved: all_embeddings.npy


## PHASE 3 — Build Book-Movie Pairs [cite: 191]
**Run On**: LOCAL MACHINE (CPU) [cite: 193]
Matches CMU book titles to TMDB movie titles using fuzzy string matching. [cite: 194]

In [11]:
import pandas as pd
from rapidfuzz import fuzz, process

# Load the cleaned dataframes saved in Phase 1.5
cmu_clean    = pd.read_csv("cmu_cleaned.csv")
movies_clean = pd.read_csv("movies_cleaned.csv")

movie_titles = movies_clean["title"].str.lower().str.strip().tolist()

pairs = []
for i, row in cmu_clean.iterrows():
    book_title = str(row["title"]).lower().strip()
    result = process.extractOne(
        book_title, movie_titles, scorer=fuzz.token_sort_ratio
    )
    if result is None: continue
    match, score, idx = result
    if score > 85:   # Only high-confidence matches
        pairs.append({
            "book_title":   row["title"],
            "movie_title":  movies_clean.iloc[idx]["title"],
            "book_summary": row["summary"],
            "movie_plot":   movies_clean.iloc[idx]["overview"],
            "book_genres":  row["mapped_genres"],
            "movie_genres": movies_clean.iloc[idx]["mapped_genres"],
            "match_score":  score
        })

pairs_df = pd.DataFrame(pairs)
print(f"Matched book-movie pairs: {len(pairs_df)}")  # Expect 400-1500
pairs_df.to_csv("book_movie_pairs.csv", index=False)
print("Saved: book_movie_pairs.csv")


Matched book-movie pairs: 2392
Saved: book_movie_pairs.csv


## PHASE 4 — Kaggle Notebook Setup [cite: 226]
**Run On**: KAGGLE [cite: 228]
Install missing packages and load local dataset into memory. [cite: 227]
Make sure to upload all 6 generated local files to Kaggle before proceeding. [cite: 225]

In [ ]:
# CELL 1 — Run this first, every session. Takes ~3 minutes.
import subprocess, sys

pkgs = [
"transformers>=4.48.0",
"accelerate>=1.2.0",
"evaluate",
"datasets>=3.1.0",
"sentence-transformers",
"bertopic",
"umap-learn",
"hdbscan",
"iterative-stratification",
"rapidfuzz",
]
for pkg in pkgs:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
# Expected: Tesla T4, ~15.8 GB


In [ ]:
# CELL 2 — Load data. Run after Cell 1.
import pickle, numpy as np

DATA = "/kaggle/input/nlp-genre-project-data/"
SAVE = "/kaggle/working/"

# Load splits
with open(f"{DATA}data_splits.pkl", "rb") as f:
splits = pickle.load(f)
X_train, y_train = splits["X_train"], splits["y_train"]
X_val,   y_val   = splits["X_val"],   splits["y_val"]
X_test,  y_test  = splits["X_test"],  splits["y_test"]

# Load embeddings (used in Phase 5 and Phase 6)
embeddings = np.load(f"{DATA}all_embeddings.npy")

# Load label binarizer
with open(f"{DATA}mlb.pkl", "rb") as f:
mlb = pickle.load(f)

TARGET_GENRES = list(mlb.classes_)
NUM_LABELS    = len(TARGET_GENRES)  # 12

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
print(f"Labels: {NUM_LABELS} genres")
print(f"Embeddings: {embeddings.shape}")


## PHASE 5 — Baseline Model: DistilBERT [cite: 284]
**Run On**: KAGGLE | ~35 minutes on T4 [cite: 286]

### 5.1 — Shared Dataset Class and Utilities [cite: 288]
Reused for DistilBERT, DeBERTa, and Longformer. [cite: 290]

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, hamming_loss, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

class GenreDataset(Dataset):
def __init__(self, texts, labels, tokenizer, max_len=512):
self.texts     = texts
self.labels    = labels
self.tokenizer = tokenizer
self.max_len   = max_len

def __len__(self): return len(self.texts)

def __getitem__(self, idx):
enc = self.tokenizer(
self.texts[idx],
max_length=self.max_len,
padding="max_length",
truncation=True,
return_tensors="pt"
)
return {
"input_ids":      enc["input_ids"].squeeze(0),
"attention_mask": enc["attention_mask"].squeeze(0),
"labels":         torch.FloatTensor(self.labels[idx])
}

def get_pos_weight(y_train, device):
"""Class weights to handle label imbalance."""
counts = y_train.sum(axis=0)
weight = (len(y_train) - counts) / (counts + 1e-8)
return torch.FloatTensor(weight).to(device)

def tune_threshold(model, loader, device):
"""Find best decision threshold on validation set."""
model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
for batch in loader:
logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
all_probs.append(torch.sigmoid(logits).cpu().numpy())
all_labels.append(batch["labels"].numpy())
probs  = np.vstack(all_probs)
labels = np.vstack(all_labels)
best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.2, 0.70, 0.05):
preds = (probs > t).astype(int)
f1 = f1_score(labels, preds, average="macro", zero_division=0)
if f1 > best_f1: best_t, best_f1 = t, f1
print(f"Best threshold: {best_t:.2f}  Val Macro-F1: {best_f1:.4f}")
return best_t

def evaluate(model, loader, threshold, model_name, device):
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
for batch in loader:
logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
all_preds.append((torch.sigmoid(logits) > threshold).cpu().numpy())
all_labels.append(batch["labels"].numpy())
P, L = np.vstack(all_preds), np.vstack(all_labels)
print(f"\n=== {model_name} Test Results (threshold={threshold}) ===")
print(f"Micro-F1:     {f1_score(L,P,average='micro',zero_division=0):.4f}")
print(f"Macro-F1:     {f1_score(L,P,average='macro',zero_division=0):.4f}")
print(f"Hamming Loss: {hamming_loss(L,P):.4f}")
print(classification_report(L, P, target_names=TARGET_GENRES, zero_division=0))


### 5.2 — DistilBERT Model Definition [cite: 357]
Projects the [CLS] token onto 12 outputs independently. [cite: 358]

In [ ]:
from transformers import AutoTokenizer, AutoModel

class DistilBertClassifier(nn.Module):
def __init__(self):
super().__init__()
self.bert = AutoModel.from_pretrained("distilbert-base-uncased")
self.drop = nn.Dropout(0.3)
self.clf  = nn.Linear(768, NUM_LABELS)

def forward(self, input_ids, attention_mask):
out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
cls = out.last_hidden_state[:, 0, :]  # [CLS] token
return self.clf(self.drop(cls))        # raw logits — sigmoid in loss


### 5.3 — Train DistilBERT [cite: 370]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train_ds = GenreDataset(X_train, y_train, tokenizer)
val_ds   = GenreDataset(X_val,   y_val,   tokenizer)
test_ds  = GenreDataset(X_test,  y_test,  tokenizer)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

model     = DistilBertClassifier().to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=get_pos_weight(y_train, device))
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer,
num_warmup_steps=int(0.1 * len(train_loader) * 5),
num_training_steps=len(train_loader) * 5)

best_macro = 0.0
for epoch in range(5):
model.train()
for batch in train_loader:
optimizer.zero_grad()
logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
loss   = criterion(logits, batch["labels"].to(device))
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
scheduler.step()
# Validate
model.eval()
all_p, all_l = [], []
with torch.no_grad():
for b in val_loader:
logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
all_p.append((torch.sigmoid(logits) > 0.5).cpu().numpy())
all_l.append(b["labels"].numpy())
macro = f1_score(np.vstack(all_l), np.vstack(all_p), average="macro", zero_division=0)
print(f"Epoch {epoch+1}/5  Val Macro-F1: {macro:.4f}")
if macro > best_macro:
best_macro = macro
torch.save(model.state_dict(), f"{SAVE}distilbert_best.pt")
print(f"  -> Saved best model")

# Tune threshold and evaluate on test set
model.load_state_dict(torch.load(f"{SAVE}distilbert_best.pt"))
best_t = tune_threshold(model, val_loader, device)
evaluate(model, test_loader, best_t, "DistilBERT", device)


## PHASE 6 — SOTA Model: DeBERTa-v3-base [cite: 414]
**Run On**: KAGGLE | ~85 minutes on T4 [cite: 416]
Uses disentangled attention to improve upon the baseline. [cite: 415]

In [ ]:
from torch.cuda.amp import autocast, GradScaler

MODEL_NAME = "microsoft/deberta-v3-base"

class DeBERTaClassifier(nn.Module):
def __init__(self):
super().__init__()
self.deberta = AutoModel.from_pretrained(MODEL_NAME)
self.drop = nn.Dropout(0.1)
self.clf  = nn.Sequential(
nn.Linear(768, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, NUM_LABELS)
)
def forward(self, input_ids, attention_mask):
out = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
return self.clf(self.drop(out.last_hidden_state[:, 0, :]))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds  = GenreDataset(X_train, y_train, tokenizer)
val_ds    = GenreDataset(X_val,   y_val,   tokenizer)
test_ds   = GenreDataset(X_test,  y_test,  tokenizer)

# Smaller batch — DeBERTa is larger than DistilBERT
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=2)

GRAD_ACCUM = 2   # Effective batch = 32
model      = DeBERTaClassifier().to(device)
scaler     = GradScaler()
criterion  = nn.BCEWithLogitsLoss(pos_weight=get_pos_weight(y_train, device))
optimizer  = AdamW(model.parameters(), lr=1e-5, weight_decay=0.01, eps=1e-6)
scheduler  = get_linear_schedule_with_warmup(optimizer,
num_warmup_steps=int(0.06 * (len(train_loader)//GRAD_ACCUM) * 4),
num_training_steps=(len(train_loader)//GRAD_ACCUM) * 4)

best_macro = 0.0
for epoch in range(4):
model.train()
optimizer.zero_grad()
for step, batch in enumerate(train_loader):
with autocast():
logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
loss   = criterion(logits, batch["labels"].to(device)) / GRAD_ACCUM
scaler.scale(loss).backward()
if (step + 1) % GRAD_ACCUM == 0:
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(optimizer)
scaler.update()
scheduler.step()
optimizer.zero_grad()
# Validate
model.eval()
all_p, all_l = [], []
with torch.no_grad():
for b in val_loader:
logits = model(b["input_ids"].to(device), b["attention_mask"].to(device))
all_p.append((torch.sigmoid(logits) > 0.5).cpu().numpy())
all_l.append(b["labels"].numpy())
macro = f1_score(np.vstack(all_l), np.vstack(all_p), average="macro", zero_division=0)
print(f"Epoch {epoch+1}/4  Val Macro-F1: {macro:.4f}")
if macro > best_macro:
best_macro = macro
torch.save(model.state_dict(), f"{SAVE}deberta_best.pt")
print(f"  -> Saved best model")

model.load_state_dict(torch.load(f"{SAVE}deberta_best.pt"))
best_t = tune_threshold(model, val_loader, device)
evaluate(model, test_loader, best_t, "DeBERTa-v3-base", device)


## PHASE 7 — Long Context Model: Longformer (Books Only) [cite: 480]
**Run On**: KAGGLE | ~90 minutes on T4 [cite: 482]
Utilizes sliding window attention to process up to 4096 tokens, specifically used here for longer CMU book summaries. [cite: 481]

In [ ]:
from transformers import LongformerTokenizer, LongformerModel

# Filter to book sources only
# We need source labels — reload from splits metadata
with open(f"{DATA}data_splits.pkl", "rb") as f:
splits = pickle.load(f)

# If source column was saved in splits, use it. Otherwise re-filter by text length.
# Book summaries are generally longer — use texts >200 words as proxy for books
book_mask_train = np.array([len(t.split()) > 80 for t in X_train])
book_mask_val   = np.array([len(t.split()) > 80 for t in X_val])
book_mask_test  = np.array([len(t.split()) > 80 for t in X_test])

Xb_train, yb_train = X_train[book_mask_train], y_train[book_mask_train]
Xb_val,   yb_val   = X_val[book_mask_val],     y_val[book_mask_val]
Xb_test,  yb_test  = X_test[book_mask_test],   y_test[book_mask_test]
print(f"Book subset — Train:{len(Xb_train)} Val:{len(Xb_val)} Test:{len(Xb_test)}")

lf_tok = LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")

class LongformerDataset(Dataset):
def __init__(self, texts, labels, tokenizer, max_len=2048):
self.texts, self.labels, self.tokenizer, self.max_len = texts, labels, tokenizer, max_len
def __len__(self): return len(self.texts)
def __getitem__(self, idx):
enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
padding="max_length", truncation=True, return_tensors="pt")
gam = torch.zeros_like(enc["input_ids"])
gam[:, 0] = 1  # Global attention on [CLS] — required
return {"input_ids":             enc["input_ids"].squeeze(0),
"attention_mask":        enc["attention_mask"].squeeze(0),
"global_attention_mask": gam.squeeze(0),
"labels":                torch.FloatTensor(self.labels[idx])}

class LongformerClassifier(nn.Module):
def __init__(self):
super().__init__()
self.lf   = LongformerModel.from_pretrained("allenai/longformer-base-4096")
self.drop = nn.Dropout(0.1)
self.clf  = nn.Linear(768, NUM_LABELS)
def forward(self, input_ids, attention_mask, global_attention_mask):
out = self.lf(input_ids=input_ids, attention_mask=attention_mask,
global_attention_mask=global_attention_mask)
return self.clf(self.drop(out.last_hidden_state[:, 0, :]))

lf_train = LongformerDataset(Xb_train, yb_train, lf_tok)
lf_val   = LongformerDataset(Xb_val,   yb_val,   lf_tok)
lf_test  = LongformerDataset(Xb_test,  yb_test,  lf_tok)

# Small batch — Longformer with max_len=2048 uses ~8GB VRAM
lf_train_loader = DataLoader(lf_train, batch_size=4, shuffle=True,  num_workers=2)
lf_val_loader   = DataLoader(lf_val,   batch_size=4, shuffle=False, num_workers=2)
lf_test_loader  = DataLoader(lf_test,  batch_size=4, shuffle=False, num_workers=2)

lf_model = LongformerClassifier().to(device)
lf_model.lf.gradient_checkpointing_enable()  # Saves VRAM at cost of ~30% speed

lf_criterion = nn.BCEWithLogitsLoss(pos_weight=get_pos_weight(yb_train, device))
lf_optimizer = AdamW(lf_model.parameters(), lr=1e-5, weight_decay=0.01)
lf_scheduler = get_linear_schedule_with_warmup(lf_optimizer,
num_warmup_steps=int(0.06 * (len(lf_train_loader)//8) * 3),
num_training_steps=(len(lf_train_loader)//8) * 3)
lf_scaler = GradScaler()

best_macro = 0.0
for epoch in range(3):
lf_model.train()
lf_optimizer.zero_grad()
for step, batch in enumerate(lf_train_loader):
with autocast():
logits = lf_model(
batch["input_ids"].to(device),
batch["attention_mask"].to(device),
batch["global_attention_mask"].to(device)
)
loss = lf_criterion(logits, batch["labels"].to(device)) / 8
lf_scaler.scale(loss).backward()
if (step + 1) % 8 == 0:
lf_scaler.unscale_(lf_optimizer)
torch.nn.utils.clip_grad_norm_(lf_model.parameters(), 1.0)
lf_scaler.step(lf_optimizer)
lf_scaler.update()
lf_scheduler.step()
lf_optimizer.zero_grad()
# Quick val check
lf_model.eval()
all_p, all_l = [], []
with torch.no_grad():
for b in lf_val_loader:
logits = lf_model(b["input_ids"].to(device), b["attention_mask"].to(device), b["global_attention_mask"].to(device))
all_p.append((torch.sigmoid(logits) > 0.5).cpu().numpy())
all_l.append(b["labels"].numpy())
macro = f1_score(np.vstack(all_l), np.vstack(all_p), average="macro", zero_division=0)
print(f"Epoch {epoch+1}/3  Val Macro-F1: {macro:.4f}")
if macro > best_macro:
best_macro = macro
torch.save(lf_model.state_dict(), f"{SAVE}longformer_best.pt")

lf_model.load_state_dict(torch.load(f"{SAVE}longformer_best.pt"))

# Custom evaluate for Longformer (needs global_attention_mask)
lf_model.eval()
all_p, all_l = [], []
with torch.no_grad():
for b in lf_test_loader:
logits = lf_model(b["input_ids"].to(device), b["attention_mask"].to(device), b["global_attention_mask"].to(device))
all_p.append((torch.sigmoid(logits) > 0.5).cpu().numpy())
all_l.append(b["labels"].numpy())
P, L = np.vstack(all_p), np.vstack(all_l)
print(f"Longformer Test Macro-F1: {f1_score(L,P,average='macro',zero_division=0):.4f}")


## PHASE 8 — Topic Modeling + Feature Fusion (Extra Task 1) [cite: 582]
**Run On**: KAGGLE | ~60 minutes [cite: 584]

### 8.1 — Train BERTopic [cite: 585]
Clusters embeddings into themes and concatenates the topic distributions as extra features for DeBERTa. [cite: 583]

In [ ]:
# IMPORTANT: Import from standalone hdbscan package, NOT sklearn.cluster
# Verify it is installed:
import pkg_resources
print(pkg_resources.get_distribution("hdbscan").version)  # Should print e.g. 0.8.41

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN   # standalone package — NOT from sklearn.cluster

# Reconstruct all_texts in the same order used in Phase 2
all_texts = list(X_train) + list(X_val) + list(X_test)

# embeddings was loaded in Phase 4 Cell 2 — already in memory
assert len(all_texts) == len(embeddings), "Count mismatch — check Phase 4 loading"

sbert = SentenceTransformer("all-MiniLM-L6-v2")

# prediction_data=True: required for .transform() calls on new texts later
hdbscan_model = HDBSCAN(
min_cluster_size=15,
metric="euclidean",
cluster_selection_method="eom",
prediction_data=True   # Only works with standalone hdbscan
)

umap_model = UMAP(n_neighbors=15, n_components=2, metric="cosine", random_state=42)

topic_model = BERTopic(
embedding_model=sbert,
umap_model=umap_model,
hdbscan_model=hdbscan_model,
nr_topics=50,
verbose=True
)

# Pass pre-computed embeddings — avoids re-encoding all texts
topics, probs = topic_model.fit_transform(all_texts, embeddings)
print(f"Topics found: {topic_model.get_topic_info().shape[0]}")

# Inspect top topics to understand what was discovered
for tid in range(min(10, topic_model.get_topic_info().shape[0])):",
words = topic_model.get_topic(tid)
if words:
print(f"Topic {tid:>3}: {", ".join([w for w,_ in words[:6]])}")
# Topic -1 is the outlier bucket — normal, ignore it


### 8.2 — Extract Topic Probability Distributions [cite: 625]
Assigns a probability score vector to each text. [cite: 626]

In [ ]:
# Split embeddings back into train / val / test
n_train = len(X_train)
n_val   = len(X_val)

emb_train = embeddings[:n_train]
emb_val   = embeddings[n_train : n_train + n_val]
emb_test  = embeddings[n_train + n_val :]

# .transform() uses prediction_data internally — needs standalone hdbscan
_, probs_train = topic_model.transform(list(X_train), emb_train)
_, probs_val   = topic_model.transform(list(X_val),   emb_val)
_, probs_test  = topic_model.transform(list(X_test),  emb_test)

print(f"probs_train: {probs_train.shape}")  # (n_train, n_topics)

np.save(f"{SAVE}topic_probs_train.npy", probs_train)
np.save(f"{SAVE}topic_probs_val.npy",   probs_val)
np.save(f"{SAVE}topic_probs_test.npy",  probs_test)


### 8.3 — DeBERTa + Topic Feature Fusion [cite: 641]
Concatenates the DeBERTa [CLS] embedding with BERTopic probabilities before passing through a final linear layer. [cite: 642]

In [ ]:
from torch.utils.data import Dataset as TorchDataset

num_topics = probs_train.shape[1]

class GenreDatasetWithTopics(TorchDataset):
def __init__(self, texts, labels, topic_probs, tokenizer, max_len=512):
self.texts, self.labels = texts, labels
self.topic_probs = topic_probs
self.tokenizer, self.max_len = tokenizer, max_len
def __len__(self): return len(self.texts)
def __getitem__(self, idx):
enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
padding="max_length", truncation=True, return_tensors="pt")
return {"input_ids":      enc["input_ids"].squeeze(0),
"attention_mask": enc["attention_mask"].squeeze(0),
"topic_probs":    torch.FloatTensor(self.topic_probs[idx]),
"labels":         torch.FloatTensor(self.labels[idx])}

class DeBERTaWithTopics(nn.Module):
def __init__(self):
super().__init__()
self.deberta = AutoModel.from_pretrained("microsoft/deberta-v3-base")
self.drop = nn.Dropout(0.1)
# [CLS] (768) + topic_probs (n_topics) concatenated before classifier
self.clf = nn.Sequential(
nn.Linear(768 + num_topics, 256), nn.GELU(),
nn.Dropout(0.1), nn.Linear(256, NUM_LABELS)
)
def forward(self, input_ids, attention_mask, topic_probs):
cls = self.deberta(input_ids=input_ids,
attention_mask=attention_mask).last_hidden_state[:, 0, :]
return self.clf(torch.cat([self.drop(cls), topic_probs], dim=-1))

deb_tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
tr_ds = GenreDatasetWithTopics(X_train, y_train, probs_train, deb_tok)
vl_ds = GenreDatasetWithTopics(X_val,   y_val,   probs_val,   deb_tok)
te_ds = GenreDatasetWithTopics(X_test,  y_test,  probs_test,  deb_tok)

tr_ld = DataLoader(tr_ds, batch_size=16, shuffle=True,  num_workers=2)
vl_ld = DataLoader(vl_ds, batch_size=16, shuffle=False, num_workers=2)
te_ld = DataLoader(te_ds, batch_size=16, shuffle=False, num_workers=2)

dt_model   = DeBERTaWithTopics().to(device)
dt_scaler  = GradScaler()
dt_crit    = nn.BCEWithLogitsLoss(pos_weight=get_pos_weight(y_train, device))
dt_optim   = AdamW(dt_model.parameters(), lr=1e-5, weight_decay=0.01)
dt_sched   = get_linear_schedule_with_warmup(dt_optim,
num_warmup_steps=int(0.06*(len(tr_ld)//2)*4),
num_training_steps=(len(tr_ld)//2)*4)

best_macro = 0.0
for epoch in range(4):
dt_model.train()
dt_optim.zero_grad()
for step, batch in enumerate(tr_ld):
with autocast():
logits = dt_model(
batch["input_ids"].to(device),
batch["attention_mask"].to(device),
batch["topic_probs"].to(device)
)
loss = dt_crit(logits, batch["labels"].to(device)) / 2
dt_scaler.scale(loss).backward()
if (step+1) % 2 == 0:
dt_scaler.unscale_(dt_optim)
torch.nn.utils.clip_grad_norm_(dt_model.parameters(), 1.0)
dt_scaler.step(dt_optim)
dt_scaler.update()
dt_sched.step()
dt_optim.zero_grad()
dt_model.eval()
all_p, all_l = [], []
with torch.no_grad():
for b in vl_ld:
logits = dt_model(b["input_ids"].to(device), b["attention_mask"].to(device), b["topic_probs"].to(device))
all_p.append((torch.sigmoid(logits) > 0.5).cpu().numpy())
all_l.append(b["labels"].numpy())
macro = f1_score(np.vstack(all_l), np.vstack(all_p), average="macro", zero_division=0)
print(f"Epoch {epoch+1}/4  Val Macro-F1: {macro:.4f}")
if macro > best_macro:
best_macro = macro
torch.save(dt_model.state_dict(), f"{SAVE}deberta_topics_best.pt")


## PHASE 9 — Mismatch Detection (Extra Task 2) [cite: 718]
**Run On**: KAGGLE | ~30-40 minutes [cite: 720]

Quantifies narrative divergence when books become films by combining three independent signals: semantic, genre, and topic divergence. [cite: 719]

### 9.1 — Load Pairs [cite: 721]

In [ ]:
import pandas as pd

pairs_df = pd.read_csv(f"{DATA}book_movie_pairs.csv")
print(f"Pairs loaded: {len(pairs_df)}")
print(pairs_df[["book_title","movie_title","match_score"]].head(10))


### 9.2 — Signal 1: Semantic Mismatch via Cross-Encoder [cite: 727]
A cross-encoder processes both summaries together to detect changes in narrative. [cite: 728, 729]

In [ ]:
from sentence_transformers import CrossEncoder

cross_enc = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

sentence_pairs = list(zip(
pairs_df["book_summary"].tolist(),
pairs_df["movie_plot"].tolist()
))

raw_scores = cross_enc.predict(sentence_pairs, show_progress_bar=True)
import torch as th
sim_scores = th.sigmoid(th.tensor(raw_scores)).numpy()

pairs_df["semantic_similarity"] = sim_scores
pairs_df["semantic_mismatch"]   = 1 - sim_scores


### 9.3 — Signal 2: Genre Label Divergence [cite: 741]
Calculates Jaccard distance between the two genre sets directly. [cite: 742]

In [ ]:
import ast

def genre_jaccard(ga, gb):
try: sa = set(ast.literal_eval(str(ga)))
except: sa = set()
try: sb = set(ast.literal_eval(str(gb)))
except: sb = set()
if not sa and not sb: return 1.0
if not sa or not sb:  return 0.0
return len(sa & sb) / len(sa | sb)

pairs_df["genre_similarity"] = pairs_df.apply(
lambda r: genre_jaccard(r["book_genres"], r["movie_genres"]), axis=1)
pairs_df["genre_mismatch"] = 1 - pairs_df["genre_similarity"]


### 9.4 — Signal 3: Topic Distribution Divergence [cite: 755]
Uses symmetric Jensen-Shannon divergence to measure shifts in themes. [cite: 756]

In [ ]:
from scipy.spatial.distance import jensenshannon

def topic_js_div(text_a, text_b, topic_model, sbert_model):
ea = sbert_model.encode([text_a])
eb = sbert_model.encode([text_b])
_, pa = topic_model.transform([text_a], ea)
_, pb = topic_model.transform([text_b], eb)
p = pa[0] + 1e-10;  p /= p.sum()
q = pb[0] + 1e-10;  q /= q.sum()
return float(jensenshannon(p, q))

# sbert and topic_model are still in memory from Phase 8
pairs_df["topic_divergence"] = pairs_df.apply(
lambda r: topic_js_div(r["book_summary"], r["movie_plot"], topic_model, sbert),
axis=1
)


### 9.5 — Composite Mismatch Score [cite: 771]
Combines the variables to produce an interpretable single output score. [cite: 772]

In [ ]:
from sklearn.preprocessing import MinMaxScaler

pairs_df["mismatch_score"] = (
0.4 * pairs_df["semantic_mismatch"] +
0.3 * pairs_df["genre_mismatch"]    +
0.3 * pairs_df["topic_divergence"]
)

scaler = MinMaxScaler()
pairs_df["mismatch_score_100"] = scaler.fit_transform(
pairs_df[["mismatch_score"]]) * 100

print("Most adapted films (highest mismatch):")
print(pairs_df.nlargest(10, "mismatch_score_100")
[["book_title","movie_title","mismatch_score_100","genre_mismatch"]].to_string())

print("\nMost faithful adaptations (lowest mismatch):")
print(pairs_df.nsmallest(10, "mismatch_score_100")
[["book_title","movie_title","mismatch_score_100","genre_mismatch"]].to_string())

pairs_df.to_csv(f"{SAVE}mismatch_results.csv", index=False)
print("Saved: mismatch_results.csv")
